In [2]:
import pandas as pd

df_train = pd.read_csv("train.csv",sep = ',')


In [3]:
print(df_train.head())
print("Test des indexes")
print((df_train["id"][0]), df_train["Temparature"][0])

   id  Temparature  Humidity  Moisture Soil Type  Crop Type  Nitrogen  \
0   0           37        70        36    Clayey  Sugarcane        36   
1   1           27        69        65     Sandy    Millets        30   
2   2           29        63        32     Sandy    Millets        24   
3   3           35        62        54     Sandy     Barley        39   
4   4           35        58        43       Red      Paddy        37   

   Potassium  Phosphorous Fertilizer Name  
0          4            5           28-28  
1          6           18           28-28  
2         12           16        17-17-17  
3         12            4        10-26-26  
4          2           16             DAP  
Test des indexes
0 37


In [4]:
import collections as coll
counter = coll.Counter(df_train["Fertilizer Name"])
print(counter)
top_three = counter.most_common(3) 
print(top_three[0])
most_fertilize_3 = [top_three[i][0] for i in range(3)]
print(most_fertilize_3)

Counter({'14-35-14': 114436, '10-26-26': 113887, '17-17-17': 112453, '28-28': 111158, '20-20': 110889, 'DAP': 94860, 'Urea': 92317})
('14-35-14', 114436)
['14-35-14', '10-26-26', '17-17-17']


In [5]:
df_test = pd.read_csv("test.csv",sep = ',')
print(df_test.head())
print("Test des indexes")
print((df_test["id"][0]), df_test["Temparature"][0])

       id  Temparature  Humidity  Moisture Soil Type    Crop Type  Nitrogen  \
0  750000           31        70        52     Sandy        Wheat        34   
1  750001           27        62        45       Red    Sugarcane        30   
2  750002           28        72        28    Clayey  Ground Nuts        14   
3  750003           37        53        57     Black  Ground Nuts        18   
4  750004           31        55        32       Red       Pulses        13   

   Potassium  Phosphorous  
0         11           24  
1         14           15  
2         15            4  
3         17           36  
4         19           14  
Test des indexes
750000 31


In [6]:
fertilizer_prediction = {}
for id in df_test["id"]:
    fertilizer_prediction[id] = most_fertilize_3

In [ ]:
import pandas as pd
import numpy as np

def mean_and_variance_data(df, numeric_columns):
    stats = {}
    for col in numeric_columns:
        mean = df[col].mean()
        std = df[col].std()
        stats[col] = (mean, std)
        
    return stats


In [7]:
def formatage_submission(fertilizer_prediction):
    df_prediction = pd.DataFrame({
    "id": fertilizer_prediction.keys(),
    "Fertilizer Name": [
        " ".join(v) for v in fertilizer_prediction.values()
    ]
    })
    return df_prediction
    


In [8]:
df_submission = formatage_submission(fertilizer_prediction)
print(df_submission.head())
print(df_submission["id"][0:3], df_submission["Fertilizer Name"][0:3])
df_submission.to_csv("submission_test.csv", index=False)
#Score : 0.27940

       id             Fertilizer Name
0  750000  14-35-14 10-26-26 17-17-17
1  750001  14-35-14 10-26-26 17-17-17
2  750002  14-35-14 10-26-26 17-17-17
3  750003  14-35-14 10-26-26 17-17-17
4  750004  14-35-14 10-26-26 17-17-17
0    750000
1    750001
2    750002
Name: id, dtype: int64 0    14-35-14 10-26-26 17-17-17
1    14-35-14 10-26-26 17-17-17
2    14-35-14 10-26-26 17-17-17
Name: Fertilizer Name, dtype: object


In [ ]:
n_travail = 2000
df_travail = ( df_train.sample(n=n_travail, random_state=2).reset_index(drop=True)
)
X_travail = df_travail.drop(columns=["Fertilizer Name"])
y_travail = df_travail["Fertilizer Name"]
from sklearn.model_selection import train_test_split

X_train_travail, X_val_travail, y_train_travail, y_val_travail = train_test_split(
    X_travail,
    y_travail,
    test_size=0.2,
    stratify=y_travail,      # très important en classification
    random_state=10
)

In [16]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("train.csv")

numeric_cols = [
    "Temparature", "Humidity", "Moisture",
    "Nitrogen", "Phosphorous", "Potassium"
]

#for col in numeric_cols:
    #plt.figure()
    #plt.hist(df[col], bins=30)
    #plt.xlabel(col)
    #plt.ylabel("Frequency")
    #plt.title(f"Distribution of {col}")
    #plt.show()


In [ ]:
def normalize_with_stats(X, stats):
    """
    Normalise X en utilisant les moyennes et écarts-types fournis.
    
    Returns:
        X_norm : DataFrame normalisé
    """
    X_norm = X.copy()
    
    for col, (mean, std) in stats.items():
        if col in X_norm.columns:
            if std != 0:
                X_norm[col] = (X_norm[col] - mean) / std
            else:
                X_norm[col] = 0.0  # cas dégénéré
    
    return X_norm


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import sklearn.pipeline

pipeline = sklearn.pipeline.Pipeline([
    ('random_forest', RandomForestClassifier(bootstrap=True, max_samples = 0.5))
])  

hyperparameters = {
    'random_forest__n_estimators': [50, 100, 200],
    'random_forest__max_depth': [5, 10,20],
    'random_forest__min_samples_leaf': [2, 5, 10]
}

gridsearch = GridSearchCV(pipeline, hyperparameters,cv=10)
gridsearch.fit(normalize_with_stats(X_train_travail), y_train_travail)
best_param = gridsearch.best_params_
print(best_param)
    

ValueError: 
All the 270 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\hugue\anaconda3\envs\mlclass\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\hugue\anaconda3\envs\mlclass\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\hugue\anaconda3\envs\mlclass\lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "c:\Users\hugue\anaconda3\envs\mlclass\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args